## Pull in UDFs

In [51]:
%run nb_udfs

StatementMeta(, e2f3c726-a1ad-4775-94c0-d7f3b0fcdccd, 93, Finished, Available, Finished, True)

## Parameters

In [52]:
workspace = 'Fabric%20of%20Middle-Earth' #have to escape any & symbols with %26 and spaces with %20
lakehouse = 'lh_capacity_management'
log_table = 'logForcedCancelledRefreshes'
max_minutes = 5 #threshold for how long a refresh can run before being cancelled
excluded_skus = ['PPU', 'FT'] #capacity SKUs to exclude (Premium Per User, Fabric Trial)
whitelist_workspace_ids = [] #workspace ids to never cancel refreshes for
whitelist_item_ids = [] #dataset or dataflow ids to never cancel


StatementMeta(, e2f3c726-a1ad-4775-94c0-d7f3b0fcdccd, 94, Finished, Available, Finished, False)

## Get Capacity-Backed Workspaces

In [53]:
#get all capacities and filter out excluded SKUs
response_cap = _base_api(
        request=f"/v1.0/myorg/admin/capacities",
        method="get"
    )
df_capacities = pd.json_normalize(response_cap.json()['value'])
df_capacities = spark.createDataFrame(df_capacities)

#filter out PPU, Fabric Trial, or any other excluded SKUs
df_valid_capacities = df_capacities.filter(~col("sku").isin(excluded_skus))
print(f'Total capacities: {df_capacities.count()}')
print(f'Valid capacities (after excluding {excluded_skus}): {df_valid_capacities.count()}')

StatementMeta(, e2f3c726-a1ad-4775-94c0-d7f3b0fcdccd, 95, Finished, Available, Finished, False)

Total capacities: 3
Valid capacities (after excluding ['PPU', 'FT']): 3


In [54]:
#get all workspaces and join to valid capacities
response = fab_client.get(f"/v1/admin/workspaces")
df_workspaces = pd.json_normalize(response.json()['workspaces'])
df_workspaces = spark.createDataFrame(df_workspaces)

#get just the valid capacity ids to join against
df_valid_cap_ids = df_valid_capacities.select(col("id").alias("validCapacityId"))

df_cap_workspaces = df_workspaces \
    .filter(df_workspaces["capacityId"].isNotNull()) \
    .filter(df_workspaces["type"] == "Workspace") \
    .filter(df_workspaces["id"] != '5d683bbb-9a09-492a-814e-2e444f53a4dd') \
    .join(df_valid_cap_ids,
          col("capacityId") == col("validCapacityId"),
          how="inner") \
    .drop("validCapacityId") \
    .withColumnRenamed("id","workspaceOgId") \
    .withColumnRenamed("name","workspaceName")

print(f'Capacity-backed workspaces (excluding {excluded_skus}): {df_cap_workspaces.count()}')

StatementMeta(, e2f3c726-a1ad-4775-94c0-d7f3b0fcdccd, 96, Finished, Available, Finished, False)

Capacity-backed workspaces (excluding ['PPU', 'FT']): 13


## Get Datasets and Dataflows

In [55]:
#get all datasets and join to capacity workspaces
response = _base_api(
        request=f"/v1.0/myorg/admin/datasets",
        method="get"
    )
df_datasets = pd.json_normalize(response.json()['value'])
df_datasets = spark.createDataFrame(df_datasets)

df_cap_datasets = df_cap_workspaces.join(
    df_datasets.filter(df_datasets["isRefreshable"] == 1),
    df_cap_workspaces["workspaceOgId"] == df_datasets["workspaceId"],
    how="inner"
).select("workspaceId","workspaceName",col("id").alias("itemId"))

print(f'Capacity datasets: {df_cap_datasets.count()}')


StatementMeta(, e2f3c726-a1ad-4775-94c0-d7f3b0fcdccd, 97, Finished, Available, Finished, False)

Capacity datasets: 59


In [56]:
#get all dataflows and join to capacity workspaces
response = _base_api(
        request=f"/v1.0/myorg/admin/dataflows",
        method="get"
    )
df_dataflows = pd.json_normalize(response.json()['value'])
df_dataflows = spark.createDataFrame(df_dataflows)

df_cap_dataflows = df_cap_workspaces.join(
    df_dataflows,
    df_cap_workspaces["workspaceOgId"] == df_dataflows["workspaceId"],
    how="inner"
).select("workspaceId","workspaceName",col("objectId").alias("itemId"))

print(f'Capacity dataflows: {df_cap_dataflows.count()}')


StatementMeta(, e2f3c726-a1ad-4775-94c0-d7f3b0fcdccd, 98, Finished, Available, Finished, False)

Capacity dataflows: 0


## UDFs: Get Running Refreshes

In [57]:
def get_running_dataset_refresh(row):
    workspaceId = row['workspaceId']
    datasetId = row['itemId']
    try:
        response = _base_api(
            request=f"/v1.0/myorg/groups/{workspaceId}/datasets/{datasetId}/refreshes",
            method="get"
        )
    except Exception as e:
        print(f'Failed api call for dataset {datasetId}: {e}')
        raise e

    json_data = response.json()
    if 'value' in json_data and isinstance(json_data['value'], list) and len(json_data['value']) > 0:
        df_pd = pd.json_normalize(json_data['value'])
        df_spark = spark.createDataFrame(df_pd)
        if "endTime" not in df_spark.columns:
            df_spark = df_spark.withColumn("endTime", lit(None))
        df_spark = df_spark.filter(
            (col("endTime").isNull()) | (col("endTime") == "")
        )
        if df_spark.count() > 0:
            df_spark = df_spark.withColumn("itemId", lit(datasetId))
            df_spark = df_spark.withColumn("workspaceId", lit(workspaceId))
            df_spark = df_spark.withColumn("itemType", lit("Dataset"))
            df_spark = df_spark.select("itemType","workspaceId","itemId",col("requestId").alias("refreshId"),"startTime","status")
            return df_spark
    return None


StatementMeta(, e2f3c726-a1ad-4775-94c0-d7f3b0fcdccd, 99, Finished, Available, Finished, False)

In [58]:
def get_running_dataflow_refresh(row):
    workspaceId = row['workspaceId']
    dataflowId = row['itemId']
    try:
        response = _base_api(
            request=f"/v1.0/myorg/groups/{workspaceId}/dataflows/{dataflowId}/transactions",
            method="get"
        )
    except Exception as e:
        print(f'Failed api call for dataflow {dataflowId}: {e}')
        raise e

    json_data = response.json()
    if 'value' in json_data and isinstance(json_data['value'], list) and len(json_data['value']) > 0:
        df_pd = pd.json_normalize(json_data['value'])
        df_spark = spark.createDataFrame(df_pd)
        if "endTime" not in df_spark.columns:
            df_spark = df_spark.withColumn("endTime", lit(None))
        df_spark = df_spark.filter(
            (col("endTime").isNull()) | (col("endTime") == "")
        )
        if df_spark.count() > 0:
            df_spark = df_spark.withColumn("itemId", lit(dataflowId))
            df_spark = df_spark.withColumn("workspaceId", lit(workspaceId))
            df_spark = df_spark.withColumn("itemType", lit("Dataflow"))
            df_spark = df_spark.select("itemType","workspaceId","itemId",col("id").alias("refreshId"),"startTime","status")
            return df_spark
    return None


StatementMeta(, e2f3c726-a1ad-4775-94c0-d7f3b0fcdccd, 100, Finished, Available, Finished, False)

## Batch Get Running Refreshes

In [59]:
dataset_rows = df_cap_datasets.collect()
dataflow_rows = df_cap_dataflows.collect()

running_refresh_list = []

with ThreadPoolExecutor(max_workers=10) as executor:
    futures = {}
    for row in dataset_rows:
        futures[executor.submit(get_running_dataset_refresh, row)] = ('Dataset', row)
    for row in dataflow_rows:
        futures[executor.submit(get_running_dataflow_refresh, row)] = ('Dataflow', row)

    for i, future in enumerate(as_completed(futures), 1):
        item_type, row = futures[future]
        try:
            result = future.result()
            if result is not None:
                running_refresh_list.append(result)
        except Exception as e:
            print(f"❌ Failed for {item_type} {row['itemId']} in workspace {row['workspaceName']}: {e}")
        if i % 500 == 0:
            print(f"✅ Processed {i} items...")

print(f'✅ Looping complete! Found {len(running_refresh_list)} items with running refreshes.')


StatementMeta(, e2f3c726-a1ad-4775-94c0-d7f3b0fcdccd, 101, Finished, Available, Finished, False)

✅ Looping complete! Found 1 items with running refreshes.


## Filter to Long-Running Refreshes and Apply Whitelist

In [60]:
from pyspark.sql.functions import current_timestamp, to_timestamp, round as spark_round 

df_skipped = None
longest_running_row = None

if running_refresh_list:
    df_running = union_batches(running_refresh_list, batch_size=50)
    df_running = df_running.withColumn("startTimeTs", to_timestamp("startTime"))
    df_running = df_running.withColumn(
        "durationMinutes",
        spark_round((unix_timestamp(current_timestamp()) - unix_timestamp(col("startTimeTs"))) / 60, 2)
    )
    df_running.orderBy(col("durationMinutes").desc()).show(5)

    # capture the longest running item before filtering
    longest_running_row = df_running.orderBy(col("durationMinutes").desc()).first()
    
    df_long_running = df_running.filter(col("durationMinutes") >= (max_minutes))
    running_count = df_running.count()

    # separate whitelisted items before filtering them out
    ws_match = col("workspaceId").isin(whitelist_workspace_ids) if whitelist_workspace_ids else lit(False)
    item_match = col("itemId").isin(whitelist_item_ids) if whitelist_item_ids else lit(False)
    df_skipped = df_long_running.filter(ws_match | item_match)
    df_long_running = df_long_running.filter(~(ws_match | item_match))

    skipped_count = df_skipped.count()
    long_running_count = df_long_running.count()
    print(f'⚠️ Found {long_running_count} refreshes to cancel, {skipped_count} whitelisted (skipped), {running_count} currently running')
    if long_running_count > 0:
        df_long_running.show(truncate=False)
    if skipped_count > 0:
        print('Whitelisted items (will not be cancelled):')
        df_skipped.show(truncate=False)
else:
    df_long_running = None
    print('✅ No running refreshes found. Nothing to cancel.')

StatementMeta(, e2f3c726-a1ad-4775-94c0-d7f3b0fcdccd, 102, Finished, Available, Finished, False)

+--------+--------------------+--------------------+--------------------+--------------------+-------+--------------------+---------------+
|itemType|         workspaceId|              itemId|           refreshId|           startTime| status|         startTimeTs|durationMinutes|
+--------+--------------------+--------------------+--------------------+--------------------+-------+--------------------+---------------+
| Dataset|253a31af-7442-450...|54258460-dadc-414...|d9a6053d-205d-442...|2026-08-26T18:18:...|Unknown|2026-08-26 18:18:...|            1.4|
+--------+--------------------+--------------------+--------------------+--------------------+-------+--------------------+---------------+

⚠️ Found 0 refreshes to cancel, 0 whitelisted (skipped), 1 currently running


## Cancel Long-Running Refreshes

In [61]:
def cancel_refresh(row):
    item_type = row['itemType']
    workspace_id = row['workspaceId']
    item_id = row['itemId']
    refresh_id = row['refreshId']

    try:
        if item_type == 'Dataset':
            response = _base_api(
                request=f"/v1.0/myorg/groups/{workspace_id}/datasets/{item_id}/refreshes/{refresh_id}",
                method="delete"
            )
        elif item_type == 'Dataflow':
            # Gen2 transaction IDs have a $ suffix; Gen1 are plain GUIDs
            is_gen2 = '$' in str(refresh_id)
            if is_gen2:
                # Try Fabric Jobs API for Gen2 CI/CD
                try:
                    response = fab_client.post(
                        f"/v1/workspaces/{workspace_id}/items/{item_id}/jobs/instances/{refresh_id.split('$')[0]}/cancel"
                    )
                except Exception as gen2_err:
                    # Standard (non-CI/CD) Gen2 has no cancel API today
                    print(f"⚠️ Skipped Gen2 non-CI/CD {item_id}: {gen2_err}")
                    return {
                        "itemType": item_type,
                        "workspaceId": workspace_id,
                        "itemId": item_id,
                        "refreshId": refresh_id,
                        "startTime": row['startTime'],
                        "durationMinutes": row['durationMinutes'],
                        "loggedTime": dt.utcnow().isoformat(),
                        "actionStatus": f"Skipped: Gen2 non-CI/CD, no cancel API available ({gen2_err})"
                    }
            else:
                response = _base_api(
                    request=f"/v1.0/myorg/groups/{workspace_id}/dataflows/{item_id}/transactions/{refresh_id}/cancel",
                    method="post"
                )
        print(f"✅ Cancelled {item_type} {item_id} refresh {refresh_id}")
        return {
            "itemType": item_type,
            "workspaceId": workspace_id,
            "itemId": item_id,
            "refreshId": refresh_id,
            "startTime": row['startTime'],
            "durationMinutes": row['durationMinutes'],
            "loggedTime": dt.utcnow().isoformat(),
            "actionStatus": "Cancelled"
        }
    except Exception as e:
        print(f"❌ Failed to cancel {item_type} {item_id}: {e}")
        return {
            "itemType": item_type,
            "workspaceId": workspace_id,
            "itemId": item_id,
            "refreshId": refresh_id,
            "startTime": row['startTime'],
            "durationMinutes": row['durationMinutes'],
            "loggedTime": dt.utcnow().isoformat(),
            "actionStatus": f"Failed: {e}"
        }


StatementMeta(, e2f3c726-a1ad-4775-94c0-d7f3b0fcdccd, 103, Finished, Available, Finished, False)

In [62]:
cancellation_log = []

if df_long_running is not None and df_long_running.count() > 0:
    cancel_rows = df_long_running.collect()

    with ThreadPoolExecutor(max_workers=10) as executor:
        future_to_row = {
            executor.submit(cancel_refresh, row): row for row in cancel_rows
        }
        for future in as_completed(future_to_row):
            try:
                result = future.result()
                cancellation_log.append(result)
            except Exception as e:
                row = future_to_row[future]
                print(f"❌ Unexpected error cancelling {row['itemId']}: {e}")

    print(f'✅ Cancellation complete. {len(cancellation_log)} refreshes processed.')
else:
    print('✅ No long-running refreshes to cancel.')


StatementMeta(, e2f3c726-a1ad-4775-94c0-d7f3b0fcdccd, 104, Finished, Available, Finished, False)

✅ No long-running refreshes to cancel.


## Write Log (Cancelled + Skipped)

In [63]:
def load_table(df):
    if df is None:
        print(f"Skipping load (no data).")
        return
    path = udf_GetFilePath(workspace, lakehouse, log_table)

    if notebookutils.fs.exists(path):
        target = DeltaTable.forPath(spark, path)

        df.write.format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .save(path)
    else:
        df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .save(path)
        print(f"Initial load")

    print(f"{df.count()} rows written")

StatementMeta(, e2f3c726-a1ad-4775-94c0-d7f3b0fcdccd, 105, Finished, Available, Finished, False)

In [64]:

# build skipped log entries from whitelisted items

skipped_log = []
if df_skipped is not None and df_skipped.count() > 0:
    for row in df_skipped.collect():
        skipped_log.append({
            "itemType": row['itemType'],
            "workspaceId": row['workspaceId'],
            "itemId": row['itemId'],
            "refreshId": row['refreshId'],
            "startTime": row['startTime'],
            "durationMinutes": row['durationMinutes'],
            "loggedTime": dt.utcnow().isoformat(),
            "actionStatus": "Skipped"
        })

# combine cancelled and skipped into one log
full_log = cancellation_log + skipped_log

if full_log:
    df_log = spark.createDataFrame(full_log)
    log_path = udf_GetFilePath(workspace, lakehouse, log_table)
    df_log.write.format("delta").mode("append").save(log_path)
    print(f'✅ Wrote {len(cancellation_log)} cancelled + {len(skipped_log)} skipped = {len(full_log)} total records to {log_table}')
    df_log.show(truncate=False)
else:
    # no cancellations or skipped items. Log the longest running item or an NA row.
    if longest_running_row is not None:
        info_log = [{
            "itemType": longest_running_row['itemType'],
            "workspaceId": longest_running_row['workspaceId'],
            "itemId": longest_running_row['itemId'],
            "refreshId": longest_running_row['refreshId'],
            "startTime": longest_running_row['startTime'],
            "durationMinutes": longest_running_row['durationMinutes'],
            "loggedTime": dt.utcnow().isoformat(),
            "actionStatus": f"Longest running item, under threshold. Not cancelled. {running_count} item(s) running at time of check."
        }]
    else:
        info_log = [{
            "itemType": "Nothing currently running",
            "workspaceId": "NA",
            "itemId": "NA",
            "refreshId": "NA",
            "startTime": None ,
            "durationMinutes": None ,
            "loggedTime": dt.utcnow().isoformat(),
            "actionStatus": "No running items found."
        }]
    
    log_schema = StructType([
    StructField("itemType", StringType()),
    StructField("workspaceId", StringType()),
    StructField("itemId", StringType()),
    StructField("refreshId", StringType()),
    StructField("startTime", StringType()),
    StructField("durationMinutes", DoubleType()),
    StructField("loggedTime", StringType()),
    StructField("actionStatus", StringType()),
    ])

    df_log = spark.createDataFrame(info_log, schema=log_schema)

    load_table(df_log)
    print(f'✅ Logged info row to {log_table}')
    df_log.show(truncate=False)


StatementMeta(, e2f3c726-a1ad-4775-94c0-d7f3b0fcdccd, 106, Finished, Available, Finished, False)

1 rows written
✅ Logged info row to logForcedCancelledRefreshes
+--------+------------------------------------+------------------------------------+------------------------------------+-----------------------+---------------+--------------------------+-----------------------------------------------------------------------------------------+
|itemType|workspaceId                         |itemId                              |refreshId                           |startTime              |durationMinutes|loggedTime                |actionStatus                                                                             |
+--------+------------------------------------+------------------------------------+------------------------------------+-----------------------+---------------+--------------------------+-----------------------------------------------------------------------------------------+
|Dataset |253a31af-7442-450f-b93a-680a4831c265|54258460-dadc-4141-ace7-ed1fc68db879|d9a6053d-205d-4